# Corrected Null Distributions and Z scoring

This notebook adapts the VAR null-model mechanics from the official NuMIT repository for our M/W scoring workflow.

NuMIT pieces mirrored here:
- random VAR coefficients + Wishart residual covariance (`VAR_from_MI`)
- spectral-radius tuning through a sigmoid parameter and root finding to match target MI
- repeated null draws with MI checks (`Null_model_VAR`)

Scope of this run:
- tensor: spike only (`spk`)
- networks: `2class_local_hom`, `2class_fittedhet_lu`

In [4]:
# Setup
import io
import json
import sys
import hashlib
from pathlib import Path
from contextlib import redirect_stdout

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.linalg import solve_discrete_lyapunov
from scipy.optimize import root_scalar
from scipy.stats import wishart
from scipy.stats import norm, rankdata

try:
    DIAG_DIR = Path(globals()['__vsc_ipynb_file__']).resolve().parent
except Exception:
    DIAG_DIR = Path(r"C:\Users\Priya\Desktop\research project (SNN Info Theory)\Project Files\zscore_diagnostic")

PROJECT_ROOT = DIAG_DIR.parent.parent
SWEEP_DIR    = PROJECT_ROOT / 'Project Files'
WIMFO_ROOT   = PROJECT_ROOT / 'wimfo'

if str(WIMFO_ROOT) not in sys.path:
    sys.path.insert(0, str(WIMFO_ROOT))

from wimfo.W_M_Info import W_M_calculator

N_NULL       = 20
RIDGE        = 1e-2
MAX_ATTEMPTS = 900

# all_sweeps is populated in the next cell from fresh spike tensor extraction
all_sweeps = {}

print(f'Diagnostic dir : {DIAG_DIR}')
print('all_sweeps will be populated from the spike extraction cell.')


Diagnostic dir : C:\Users\Priya\Desktop\research project (SNN Info Theory)\Project Files\zscore_diagnostic
all_sweeps will be populated from the spike extraction cell.


In [8]:
# Load models, extract correct binary spike tensors, compute observed W/M sweep
# -------------------------------------------------------------------------------
# BUG FIX: the parity notebook's collect_hidden_tensor_matrix used
#   layer_recs[0][0]  for tensor_key=="spk"
# but layer_recs[0] = (syn_rec, mem_rec, spk_rec) from RecurrentSpikingLayer:
#   index 0 → syn_rec  (synaptic current, continuous)  ← was wrongly used
#   index 1 → mem_rec  (membrane potential, continuous)
#   index 2 → spk_rec  (binary 0/1 spike train)        ← correct
#
# All pre-existing JSON sweep files therefore contain W/M values derived from
# synaptic current, not spikes.  This cell re-extracts from index [2] and
# recomputes, overwriting all_sweeps with the correct observed values.
# -------------------------------------------------------------------------------

import importlib
import torch

PAPER_ROOT           = PROJECT_ROOT / 'neural_heterogeneity' / 'SuGD_code'
CHECKPOINT_DIR_PARITY = PROJECT_ROOT / 'Project Files' / 'Checkpoints' / 'Parity'
SHD_TEST_PATH        = PROJECT_ROOT / 'data' / 'shd' / 'shd_test.h5'

if str(PAPER_ROOT) not in sys.path:
    sys.path.insert(0, str(PAPER_ROOT))

RSNN   = importlib.import_module('model').RSNN
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

from data_gen import open_file
from wimfo.utils.utils_gauss import get_cov

M_INFO_SUBSET_SIZES   = [2, 4, 8, 16, 32]
OBS_MAX_BATCHES       = 2
OBS_DOWNSAMPLE_STRIDE = 4
OBS_LAG               = 1
OBS_RIDGE             = 1e-2

NETWORK_CHECKPOINT_MAP = {
    '2class_local_hom':    CHECKPOINT_DIR_PARITY / '2class_local_hom.pt',
    '2class_fittedhet_lu': CHECKPOINT_DIR_PARITY / '2class_fittedhet_loguniform.pt',
}

SPK_TENSOR_DIR = DIAG_DIR / 'spk_tensors'
SPK_TENSOR_DIR.mkdir(exist_ok=True)

print(f'Device           : {DEVICE}')
print(f'SHD test exists  : {SHD_TEST_PATH.exists()}')
for name, path in NETWORK_CHECKPOINT_MAP.items():
    print(f'  {name}: checkpoint exists={path.exists()}')


# ── Helper: load checkpoint ────────────────────────────────────────────────

def load_parity_model(ckpt_path, device):
    """Load a saved RSNN parity model and restore non-serialisable prms fields."""
    ckpt  = torch.load(ckpt_path, map_location=device)
    prms  = dict(ckpt['prms'])
    prms['dtype']  = torch.float
    prms['device'] = device
    prms['cuda']   = device.type == 'cuda'
    model = RSNN(prms, rec=True).to(device)
    model.load_state_dict(ckpt['model_state_dict'])
    model.eval()
    return model, prms


# ── Helper: collect spike tensor from index [2] ────────────────────────────

@torch.no_grad()
def collect_spike_tensor_from_file(model, prms, shd_test_path,
                                    max_batches=2, downsample_stride=4):
    """
    Run the model on SHD test batches and return layer_recs[0][2],
    the actual binary spike tensor (SuSpike outputs 0.0 and 1.0 only).

    RecurrentSpikingLayer.forward() returns (syn_rec, mem_rec, spk_rec):
      layer_recs[0][0] = syn_rec  (synaptic current, continuous)
      layer_recs[0][1] = mem_rec  (membrane potential, continuous)
      layer_recs[0][2] = spk_rec  (binary 0/1)   ← THIS is what we want
    """
    stride    = max(int(downsample_stride), 1)
    nb_hidden = int(model.network[0].output_size)

    raw_units, raw_times, raw_labels = open_file(str(shd_test_path))
    try:
        units  = list(raw_units[:])
        times  = list(raw_times[:])
        labels = np.array(raw_labels[:])
    finally:
        raw_units._v_file.close()

    batch_size = prms['batch_size']
    nb_steps   = prms['nb_steps']
    nb_inputs  = prms['nb_inputs']
    inv_dt     = 1.0 / prms['time_step']
    class_list = prms['class_list']

    sample_index = np.where(np.isin(labels, class_list))[0]
    n_batches    = min(max_batches, -(-len(sample_index) // batch_size))

    chunks = []
    for bidx in range(n_batches):
        batch_index = sample_index[
            batch_size * bidx : min(len(sample_index), batch_size * (bidx + 1))
        ]
        actual_bs = len(batch_index)

        t_arrays = [np.round(times[idx] * inv_dt).astype(np.int64) for idx in batch_index]
        u_arrays = [units[idx] for idx in batch_index]
        lengths  = np.array([len(a) for a in t_arrays], dtype=np.int64)

        if lengths.sum():
            all_ts = np.concatenate(t_arrays)
            all_us = np.concatenate(u_arrays)
            all_bc = np.repeat(np.arange(actual_bs, dtype=np.int64), lengths)
            valid  = all_ts < nb_steps
            all_ts, all_us, all_bc = all_ts[valid], all_us[valid], all_bc[valid]
            i       = torch.from_numpy(np.stack([all_bc, all_ts, all_us]))
            v       = torch.ones(all_ts.size, dtype=torch.float32)
            X_batch = torch.sparse_coo_tensor(
                i, v, torch.Size([actual_bs, nb_steps, nb_inputs])
            ).to_dense()
        else:
            X_batch = torch.zeros(actual_bs, nb_steps, nb_inputs)

        X_batch    = X_batch.clamp(max=1.0).to(DEVICE)
        layer_recs = model(0, 0, X_batch)

        # Correct index: [0][2] = spk_rec (binary 0/1)
        spk = layer_recs[0][2]                              # [batch, nb_steps, nb_hidden]
        spk = spk[:, ::stride, :].detach().cpu().numpy()   # [batch, T/s,      nb_hidden]
        spk = np.transpose(spk, (2, 0, 1)).reshape(nb_hidden, -1)
        chunks.append(spk)

    return np.concatenate(chunks, axis=1)   # [nb_hidden, total_timepoints]


# ── Helper: Gaussian copula + W/M calculation ──────────────────────────────

def _gc_normalize(data):
    """Rank-based Gaussian copula normalisation, row-wise."""
    transformed = np.zeros_like(data, dtype=np.float64)
    for idx, row in enumerate(data):
        if np.allclose(row, row[0]):
            continue
        ranks            = rankdata(row, method='average')
        uniform          = np.clip((ranks - 0.5) / len(row), 1e-6, 1.0 - 1e-6)
        transformed[idx] = norm.ppf(uniform)
    return transformed


def _inject_jitter(data, eps=1e-6):
    """Replace constant rows with tiny linear gradients so they are not degenerate."""
    row_std = np.nanstd(data, axis=1)
    deg_idx = np.flatnonzero(row_std <= 1e-12)
    if deg_idx.size == 0:
        return data
    data = data.copy()
    base = np.linspace(-1.0, 1.0, data.shape[1], dtype=np.float64)
    for offset, ridx in enumerate(deg_idx, start=1):
        data[ridx] = (eps * offset) * base
    return data


def compute_wm_from_spike_matrix(hidden_data, lag=1, ridge=1e-2):
    """
    Compute (W_bits, M_bits, n_samples) from a binary [neurons x timepoints] matrix.
    Replicates _compute_wm_from_matrix from the parity notebook: gaussian copula
    normalisation followed by wimfo's gaussian estimator, with an explicit ridge
    covariance fallback.
    """
    gdata = _gc_normalize(hidden_data.astype(np.float64).copy())
    gdata = np.nan_to_num(gdata, nan=0.0, posinf=0.0, neginf=0.0)
    gdata = _inject_jitter(gdata)

    opt_order = [
        ('Adam',   {'atol': 1e-3, 'rtol': 1e-3, 'max_iter': 30000}),
        ('Adam',   {'atol': 5e-3, 'rtol': 5e-3, 'max_iter': 60000}),
        ('Newton', None),
    ]
    lag0           = max(int(lag), 1)
    lag_cands      = sorted({lag0, lag0 * 2, lag0 * 4})
    stride_cands   = [1, 2, 4]
    ridge_cands    = sorted({float(ridge), 5.0 * float(ridge), 1e-1, 2e-1, 5e-1, 1.0})

    # Primary path: pass Gaussian-copula data directly
    for s in stride_cands:
        dv = gdata[:, ::s]
        if dv.shape[1] <= max(8, 2 * dv.shape[0] + 2):
            continue
        for lt in lag_cands:
            for opt, opts in opt_order:
                try:
                    with io.StringIO() as buf, redirect_stdout(buf):
                        w, m = W_M_calculator(dv, t=lt, option='data',
                                              type='gaussian', unit='bits',
                                              verbose=False, optimiser=opt,
                                              options=opts)
                except Exception:
                    continue
                if np.isfinite(w) and np.isfinite(m):
                    return float(w), float(m), int(dv.shape[1])

    # Fallback: explicit ridge-regularised lagged covariance
    last_err = None
    for s in stride_cands:
        dv = gdata[:, ::s]
        if dv.shape[1] <= max(8, 2 * dv.shape[0] + 2):
            continue
        for lt in lag_cands:
            try:
                cov = np.asarray(get_cov(dv, t=lt), dtype=np.float64)
            except Exception as exc:
                last_err = exc
                continue
            cov   = np.nan_to_num(cov, nan=0.0, posinf=0.0, neginf=0.0)
            cov   = 0.5 * (cov + cov.T)
            scale = np.trace(cov) / max(cov.shape[0], 1)
            if not np.isfinite(scale) or scale <= 0.0:
                scale = 1.0
            eye = np.eye(cov.shape[0], dtype=np.float64)
            for rd in ridge_cands:
                lc = cov + rd * scale * eye
                try:
                    ev, evec = np.linalg.eigh(lc)
                    ev = np.clip(ev, max(scale * 1e-8, 1e-10), None)
                    lc = (evec * ev[np.newaxis, :]) @ evec.T
                    lc = 0.5 * (lc + lc.T)
                except np.linalg.LinAlgError as exc:
                    last_err = exc
                    continue
                for opt, opts in opt_order:
                    try:
                        with io.StringIO() as buf, redirect_stdout(buf):
                            w, m = W_M_calculator(lc, option='distr',
                                                  type='gaussian', unit='bits',
                                                  verbose=False, optimiser=opt,
                                                  options=opts)
                    except Exception as exc:
                        last_err = exc
                        continue
                    if np.isfinite(w) and np.isfinite(m):
                        return float(w), float(m), int(dv.shape[1])

    raise RuntimeError(
        f'All W/M estimation paths failed. Last error: {last_err}'
    )


# ── Main: collect spikes → compute W/M → populate all_sweeps ──────────────

all_sweeps = {}

for net_key in ['2class_local_hom', '2class_fittedhet_lu']:
    ckpt_path = NETWORK_CHECKPOINT_MAP[net_key]
    print(f'\n{"=" * 60}')
    print(f'Network    : {net_key}')
    print(f'Checkpoint : {ckpt_path.name}')

    model, prms = load_parity_model(ckpt_path, DEVICE)
    print(f'Model loaded  — nb_recurrent={prms["nb_recurrent"]}, '
          f'nb_outputs={prms["nb_outputs"]}')

    # ── Extract layer_recs[0][2] = actual binary spike tensor ──────────────
    print('Collecting layer_recs[0][2]  (spk_rec, binary 0/1) ...')
    spk_tensor = collect_spike_tensor_from_file(
        model, prms, SHD_TEST_PATH,
        max_batches=OBS_MAX_BATCHES,
        downsample_stride=OBS_DOWNSAMPLE_STRIDE,
    )

    unique_vals = np.unique(spk_tensor)
    is_binary   = np.all((spk_tensor == 0.0) | (spk_tensor == 1.0))
    print(f'  Shape       : {spk_tensor.shape}')
    print(f'  Unique vals : {unique_vals}  (expected [0. 1.])')
    print(f'  Is binary   : {is_binary}')
    print(f'  Mean rate   : {float(spk_tensor.mean()):.4f}')
    if not is_binary:
        print('  WARNING: tensor contains non-binary values – check SuSpike path')

    # Save for audit
    npy_path = SPK_TENSOR_DIR / f'{net_key}_spk_tensor.npy'
    np.save(npy_path, spk_tensor.astype(np.float32))
    print(f'  Saved       : {npy_path}')

    # ── Compute observed W/M for each subset size ──────────────────────────
    nb_hidden  = int(spk_tensor.shape[0])
    sweep_rows = []

    print(f'\nObserved W/M sweep  (lag={OBS_LAG}, ridge={OBS_RIDGE}):')
    for k in M_INFO_SUBSET_SIZES:
        k   = int(k)
        idx = np.linspace(0, nb_hidden - 1, min(k, nb_hidden), dtype=int)
        try:
            w_bits, m_bits, n_samp = compute_wm_from_spike_matrix(
                spk_tensor[idx, :], lag=OBS_LAG, ridge=OBS_RIDGE,
            )
            row = {
                'subset_size':     k,
                'observed_W_bits': w_bits,
                'observed_M_bits': m_bits,
                'samples':         n_samp,
                'status':          'ok',
            }
            print(f'  k={k:2d} | W={w_bits:9.5f} bits | M={m_bits:9.5f} bits')
        except Exception as exc:
            row = {
                'subset_size':     k,
                'observed_W_bits': float('nan'),
                'observed_M_bits': float('nan'),
                'samples':         0,
                'status':          f'error: {exc}',
            }
            print(f'  k={k:2d} | ERROR: {exc}')
        sweep_rows.append(row)

    all_sweeps[(net_key, 'spk')] = sweep_rows

print(f'\n{"=" * 60}')
print('Observed W/M sweep complete.  all_sweeps summary:')
for key, rows in all_sweeps.items():
    n_ok = sum(1 for r in rows if r.get('status') == 'ok')
    print(f'  {key}: {n_ok}/{len(rows)} subset sizes OK')


Device           : cuda
SHD test exists  : True
  2class_local_hom: checkpoint exists=True
  2class_fittedhet_lu: checkpoint exists=True

Network    : 2class_local_hom
Checkpoint : 2class_local_hom.pt
Model loaded  — nb_recurrent=32, nb_outputs=2
  Shape       : (32, 128000)
  Unique vals : [0. 1.]  (expected [0. 1.])
  Is binary   : True
  Mean rate   : 0.0755
  Saved       : C:\Users\Priya\Desktop\research project (SNN Info Theory)\Project Files\zscore_diagnostic\spk_tensors\2class_local_hom_spk_tensor.npy

Observed W/M sweep  (lag=1, ridge=0.01):
  k= 2 | W=  0.46117 bits | M=  0.00039 bits
  k= 4 | W=  1.00988 bits | M=  0.00619 bits
  k= 8 | W=  4.45477 bits | M=  0.55940 bits
  k=16 | W=  4.22068 bits | M=  1.27533 bits
  k=32 | W=  6.04421 bits | M=  2.21800 bits

Network    : 2class_fittedhet_lu
Checkpoint : 2class_fittedhet_loguniform.pt
Model loaded  — nb_recurrent=32, nb_outputs=2
  Shape       : (32, 128000)
  Unique vals : [0. 1.]  (expected [0. 1.])
  Is binary   : True
 

KeyboardInterrupt: 

In [6]:
# NuMIT-adapted helpers (VAR_from_MI + Null_model_VAR mechanics)
def _sigmoid(x, M=1.0):
    x = np.clip(float(x), -60.0, 60.0)
    return float(M) / (1.0 + np.exp(-x))


def _regularize_cov(cov, ridge=1e-9):
    cov = np.asarray(cov, dtype=np.float64)
    cov = np.nan_to_num(cov, nan=0.0, posinf=0.0, neginf=0.0)
    cov = 0.5 * (cov + cov.T)
    scale = np.trace(cov) / max(cov.shape[0], 1)
    if (not np.isfinite(scale)) or scale <= 0.0:
        scale = 1.0
    return cov + float(ridge) * scale * np.eye(cov.shape[0], dtype=np.float64)


def _logdet_pd(mat):
    sign, val = np.linalg.slogdet(mat)
    if sign <= 0 or (not np.isfinite(val)):
        raise ValueError('Matrix not positive definite for logdet.')
    return float(val)


def _spectral_radius(A):
    eig = np.linalg.eigvals(A)
    return float(np.max(np.abs(eig)))


def _mi_var1_bits(A, V):
    S = solve_discrete_lyapunov(A, V)
    S = _regularize_cov(S, ridge=1e-12)
    return 0.5 * (_logdet_pd(S) - _logdet_pd(V)) / np.log(2.0)


def _lagged_cov_from_var1(A, V):
    S = solve_discrete_lyapunov(A, V)
    S = _regularize_cov(S, ridge=1e-12)
    Cpf = S @ A.T
    return np.block([[S, Cpf], [Cpf.T, S]])


def _numit_var1_from_mi(mi_target_bits, nvar, rng, g_cap=0.9995, tol_bits=1e-3):
    # random VAR template (analogous to var_rand for p=1 in this workflow)
    A0 = rng.normal(0.0, 1.0, size=(int(nvar), int(nvar)))
    sr = _spectral_radius(A0)
    if (not np.isfinite(sr)) or sr <= 1e-12:
        return None, None, True, np.nan
    A_unit = A0 / sr

    # NuMIT uses Wishart residual covariance in VAR_from_MI
    V = wishart.rvs(df=int(nvar) + 1, scale=np.eye(int(nvar)), random_state=rng)
    V = _regularize_cov(V, ridge=1e-9)

    def f_x(x):
        g = float(g_cap) * _sigmoid(x, 1.0)
        A = g * A_unit
        try:
            mi_val = _mi_var1_bits(A, V)
        except Exception:
            return np.nan
        return float(mi_val - float(mi_target_bits))

    xs = np.array([-12, -8, -5, -3, -2, -1, 0, 1, 2, 3, 5, 8, 12], dtype=np.float64)
    fs = np.asarray([f_x(x) for x in xs], dtype=np.float64)

    x_star = None
    near_idx = np.where(np.isfinite(fs) & (np.abs(fs) <= float(tol_bits)))[0]
    if near_idx.size > 0:
        x_star = float(xs[int(near_idx[0])])
    else:
        for i in range(len(xs) - 1):
            f1, f2 = fs[i], fs[i + 1]
            if not (np.isfinite(f1) and np.isfinite(f2)):
                continue
            if f1 * f2 > 0:
                continue
            try:
                sol = root_scalar(
                    f_x,
                    bracket=[float(xs[i]), float(xs[i + 1])],
                    method='brentq',
                    xtol=1e-6,
                    rtol=1e-6,
                    maxiter=100,
                )
            except Exception:
                sol = None
            if sol is not None and sol.converged and np.isfinite(sol.root):
                x_star = float(sol.root)
                break

    if x_star is None:
        return None, None, True, np.nan

    g_star = float(g_cap) * _sigmoid(x_star, 1.0)
    A = g_star * A_unit

    try:
        mi_check = _mi_var1_bits(A, V)
    except Exception:
        return None, None, True, np.nan

    if not np.isfinite(mi_check):
        return None, None, True, np.nan

    allowed_err = max(float(tol_bits), 0.02 * max(float(mi_target_bits), 1e-8))
    if abs(float(mi_check) - float(mi_target_bits)) > allowed_err:
        return None, None, True, float(mi_check)

    return A, V, False, float(mi_check)


def _wm_from_lagged_cov(lagged_cov, ridge=1e-2):
    cov = _regularize_cov(lagged_cov, ridge=float(ridge))
    with io.StringIO() as buf, redirect_stdout(buf):
        w_bits, m_bits = W_M_calculator(
            cov,
            option='distr',
            type='gaussian',
            unit='bits',
            verbose=False,
            optimiser='Newton',
            options=None,
        )
    return float(w_bits), float(m_bits)


def _stable_seed(net_key, tensor, subset_size):
    token = f'{net_key}|{tensor}|{int(subset_size)}'.encode('utf-8')
    return int(hashlib.sha256(token).hexdigest()[:8], 16)


def build_numit_null_distribution(nvar, tdmi_target_bits, n_null=20, seed=12345, ridge=1e-2, max_attempts=900):
    rng = np.random.default_rng(int(seed))

    null_m = []
    null_w = []
    model_mi_vals = []
    wm_tdmi_vals = []

    attempts = 0
    while len(null_m) < int(n_null) and attempts < int(max_attempts):
        attempts += 1

        A, V, failed, mi_model = _numit_var1_from_mi(tdmi_target_bits, nvar, rng)
        if failed:
            continue

        try:
            lag_cov = _lagged_cov_from_var1(A, V)
            w_bits, m_bits = _wm_from_lagged_cov(lag_cov, ridge=ridge)
        except Exception:
            continue

        if not (np.isfinite(w_bits) and np.isfinite(m_bits)):
            continue

        null_w.append(float(w_bits))
        null_m.append(float(m_bits))
        model_mi_vals.append(float(mi_model))
        wm_tdmi_vals.append(float(w_bits + m_bits))

    return {
        'null_M_values': null_m,
        'null_W_values': null_w,
        'model_mi_bits': model_mi_vals,
        'wm_tdmi_bits': wm_tdmi_vals,
        'n_null_valid': int(len(null_m)),
        'n_attempts': int(attempts),
    }


def score_observed_vs_null_m(observed_m, null_m_values):
    vals = np.asarray(null_m_values, dtype=np.float64)
    vals = vals[np.isfinite(vals)]
    if vals.size == 0:
        return {
            'null_mean': np.nan,
            'null_std': np.nan,
            'z': np.nan,
            'p_upper': np.nan,
            'p_lower': np.nan,
            'p_two_sided': np.nan,
        }

    mu = float(vals.mean())
    sd = float(vals.std(ddof=1)) if vals.size > 1 else np.nan
    z = float((float(observed_m) - mu) / sd) if (np.isfinite(sd) and sd > 1e-12) else np.nan

    ge = np.sum(vals >= float(observed_m))
    le = np.sum(vals <= float(observed_m))
    p_upper = float((1.0 + ge) / (vals.size + 1.0))
    p_lower = float((1.0 + le) / (vals.size + 1.0))
    p_two = float(min(1.0, 2.0 * min(p_upper, p_lower)))

    return {
        'null_mean': mu,
        'null_std': sd,
        'z': z,
        'p_upper': p_upper,
        'p_lower': p_lower,
        'p_two_sided': p_two,
    }

In [7]:
# Run scoring only for spike tensors of 2-class homogeneous and log-uniform networks
records = []

for (net_key, tensor), rows in sorted(all_sweeps.items(), key=lambda x: (x[0][0], x[0][1])):
    print(f'\nScoring {net_key} / {tensor}')

    for row in sorted(rows, key=lambda r: int(r.get('subset_size', 0))):
        k = int(row['subset_size'])
        obs_m = float(row['observed_M_bits'])
        obs_w = float(row['observed_W_bits'])
        tdmi_target = float(obs_m + obs_w)

        null_dist = build_numit_null_distribution(
            nvar=k,
            tdmi_target_bits=tdmi_target,
            n_null=N_NULL,
            seed=_stable_seed(net_key, tensor, k),
            ridge=RIDGE,
            max_attempts=MAX_ATTEMPTS,
        )

        score = score_observed_vs_null_m(obs_m, null_dist['null_M_values'])

        model_mi = np.asarray(null_dist['model_mi_bits'], dtype=np.float64)
        wm_tdmi = np.asarray(null_dist['wm_tdmi_bits'], dtype=np.float64)
        model_mi_mean = float(np.nanmean(model_mi)) if model_mi.size else np.nan
        wm_tdmi_mean = float(np.nanmean(wm_tdmi)) if wm_tdmi.size else np.nan

        records.append({
            'network': net_key,
            'tensor': tensor,
            'subset_size': k,
            'observed_M_bits': obs_m,
            'observed_W_bits': obs_w,
            'observed_TDMI_bits': tdmi_target,
            'null_M_mean': score['null_mean'],
            'null_M_std': score['null_std'],
            'M_z_tdmi_null': score['z'],
            'M_p_upper_tdmi_null': score['p_upper'],
            'M_p_lower_tdmi_null': score['p_lower'],
            'M_p_two_sided_tdmi_null': score['p_two_sided'],
            'n_null_valid': int(null_dist['n_null_valid']),
            'n_attempts': int(null_dist['n_attempts']),
            'null_model_mi_mean': model_mi_mean,
            'null_solver_tdmi_mean': wm_tdmi_mean,
            'null_M_values': null_dist['null_M_values'],
            'null_W_values': null_dist['null_W_values'],
            'model_mi_bits': null_dist['model_mi_bits'],
            'wm_tdmi_bits': null_dist['wm_tdmi_bits'],
        })

        print(
            f'  k={k:2d} | target TDMI={tdmi_target:8.4f} | ' +
            f'null_n={null_dist["n_null_valid"]:2d}/{N_NULL} | ' +
            f'z={score["z"]:7.3f} | p_upper={score["p_upper"]:.4f} | ' +
            f'model_MI_mean={model_mi_mean:8.4f} | solver_TDMI_mean={wm_tdmi_mean:8.4f}'
        )

scores_df = pd.DataFrame.from_records(records)
scores_df = scores_df.sort_values(['network', 'subset_size']).reset_index(drop=True)

show_cols = [
    'network', 'tensor', 'subset_size',
    'observed_M_bits', 'observed_TDMI_bits',
    'null_M_mean', 'null_M_std', 'M_z_tdmi_null',
    'M_p_upper_tdmi_null', 'M_p_lower_tdmi_null', 'M_p_two_sided_tdmi_null',
    'n_null_valid', 'n_attempts', 'null_model_mi_mean', 'null_solver_tdmi_mean',
]

print('\nScoring complete.')
display(scores_df[show_cols])

csv_path = DIAG_DIR / 'corrected_numit_spk_scores.csv'
scores_df.drop(columns=['null_M_values', 'null_W_values', 'model_mi_bits', 'wm_tdmi_bits']).to_csv(csv_path, index=False)
print(f'Saved summary CSV: {csv_path}')


Scoring 2class_fittedhet_lu / spk
  k= 2 | target TDMI=  0.5472 | null_n=20/20 | z= -0.853 | p_upper=0.6667 | model_MI_mean=  0.5472 | solver_TDMI_mean=  0.5132
  k= 4 | target TDMI=  3.4692 | null_n=20/20 | z= -3.498 | p_upper=1.0000 | model_MI_mean=  3.4692 | solver_TDMI_mean=  2.8527
  k= 8 | target TDMI=  4.6239 | null_n=20/20 | z= -6.291 | p_upper=1.0000 | model_MI_mean=  4.6239 | solver_TDMI_mean=  3.8329


KeyboardInterrupt: 

In [ ]:
# Quick visual summary
fig, axes = plt.subplots(1, 2, figsize=(10, 4), sharey=True)

for ax, net_key in zip(axes, ['2class_local_hom', '2class_fittedhet_lu']):
    sub = scores_df[scores_df['network'] == net_key].sort_values('subset_size')
    ax.plot(sub['subset_size'], sub['M_z_tdmi_null'], marker='o', label='z score')
    ax.axhline(0.0, color='gray', linewidth=1, linestyle='--')
    ax.set_title(net_key + ' (spk)')
    ax.set_xlabel('subset size')
    ax.set_ylabel('M z score')
    ax.grid(alpha=0.3)

plt.tight_layout()
outp = DIAG_DIR / 'corrected_numit_spk_zscores.png'
plt.savefig(outp, dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved plot: {outp}')